# NeoOLAF DocRED — resume ONLY failed documents + final micro evaluation (self-contained V2)

This notebook resumes the **existing** DocRED v6.2 full-dev experiment under the exact original batch root:

`examples/RAGTreeDatasets/runs/docred_native_v5_1_dev_streaming`

It does **not** start a new experiment.

## Intended starting state

The previous full run requested **998 DocRED dev records**:

- **978 completed**
- **20 failed**

This notebook:

1. rebuilds the current aggregate **offline** from the existing saved artifacts;
2. lists the currently failed records;
3. verifies that every non-failed document is already completed/resumable;
4. reruns the v6.2 streaming scheduler with:
   - `resume_completed=True`
   - `retry_failed_documents=True`
5. therefore **completed documents are skipped** and only failed/unresolved documents are eligible for paid rerun;
6. uses the original frozen DocRED execution settings:
   - model `openai/gpt-oss-20b`
   - `DOCUMENT_WORKERS=4`
   - `LAYER_WORKERS=16`
   - frozen v5.1 scientific profile/guidance/evaluator
7. rebuilds the final aggregate from disk afterward;
8. reports the definitive DocRED relation/entity/endpoint micro metrics and runtime.

If some failed documents remain after one invocation, rerun this notebook: successfully recovered documents are then skipped and only the remaining failures are retried.


### V2 import hotfix
This version embeds and restores the missing `docred_native_batch_v6_2_dev_streaming.py` helper automatically. It does not modify `src/neoolaf`.


In [1]:
from __future__ import annotations

import os
import sys
import json
import multiprocessing as mp
from getpass import getpass
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents, Path(r"C:\Users\galencarmedeiro\NeoOLAF")]
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the NeoOLAF repository "
        "or set the working directory to the NeoOLAF project root."
    )


def first_existing_path(label: str, candidates: list[Path]) -> Path:
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        candidate = candidate if candidate.is_absolute() else PROJECT_ROOT / candidate
        candidate = candidate.resolve()
        checked.append(candidate)
        if candidate.is_file():
            print(f"{label}={candidate}")
            return candidate
    raise FileNotFoundError(
        f"Could not find {label}. Checked:\n" + "\n".join(str(path) for path in checked)
    )


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
TOOLS_DIR.mkdir(parents=True, exist_ok=True)

for path in [PROJECT_ROOT / "src", PROJECT_ROOT, TOOLS_DIR, NOTEBOOK_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# ---------------------------------------------------------------------------
# SELF-CONTAINED v6.2 helper bootstrap
# ---------------------------------------------------------------------------
# The previous notebook assumed docred_native_batch_v6_2_dev_streaming.py
# already existed under examples/RAGTreeDatasets/tools. In your checkout it
# does not, which caused ModuleNotFoundError.
#
# This V2 notebook contains the exact v6.2 helper source and installs it into
# examples/RAGTreeDatasets/tools ONLY if it is missing. It never modifies
# anything under src/neoolaf.
V62_HELPER = TOOLS_DIR / "docred_native_batch_v6_2_dev_streaming.py"

if not V62_HELPER.is_file():
    _embedded_v62 = 'from __future__ import annotations\n\nimport csv\nimport json\nimport multiprocessing as mp\nimport statistics\nimport time\nimport traceback\nfrom concurrent.futures import FIRST_COMPLETED, ProcessPoolExecutor, wait\nfrom dataclasses import asdict\nfrom pathlib import Path\nfrom typing import Any, Iterator\n\nimport docred_native_batch_v6_1 as base\n\n\nORCHESTRATOR_VERSION = "v6.2-dev-filtered-bounded-streaming"\nEXACT_REQUIRED_TYPE = "dev"\n\nBatchRunConfig = base.BatchRunConfig\nPreparedDocument = base.PreparedDocument\nread_json = base.read_json\nwrite_json = base.write_json\nappend_jsonl = base.append_jsonl\niter_jsonl = base.iter_jsonl\nsha256_bytes = base.sha256_bytes\nsha256_file = base.sha256_file\ndocument_id = base.document_id\nsafe_slug = base.safe_slug\nstrip_gold = base.strip_gold\n\n\ndef _json_default(value: Any) -> Any:\n    return base._json_default(value)\n\n\ndef is_exact_type(record: dict[str, Any], required_type: str = EXACT_REQUIRED_TYPE) -> bool:\n    """Match only the literal JSON key ``type``; never fall back to ``split``."""\n    return str(record.get("type") or "").strip().lower() == str(required_type).strip().lower()\n\n\ndef count_exact_type_records(\n    dataset_jsonl: str | Path,\n    *,\n    required_type: str = EXACT_REQUIRED_TYPE,\n    first_n_ids: int = 5,\n) -> dict[str, Any]:\n    """Count matching records with a single streaming pass and O(1) document memory."""\n    total = 0\n    matching = 0\n    first_ids: list[str] = []\n    for source_index, record in enumerate(iter_jsonl(dataset_jsonl)):\n        total += 1\n        if not is_exact_type(record, required_type):\n            continue\n        matching += 1\n        if len(first_ids) < max(0, int(first_n_ids)):\n            first_ids.append(document_id(record, source_index))\n    return {\n        "total_records": total,\n        "matching_records": matching,\n        "required_type": required_type,\n        "first_matching_ids": first_ids,\n    }\n\n\ndef iter_prepared_documents_streaming(\n    *,\n    dataset_jsonl: str | Path,\n    task_guidance_path: str | Path,\n    batch_root: str | Path,\n    run_all_documents: bool,\n    smoke_document_limit: int = 5,\n    required_type: str = EXACT_REQUIRED_TYPE,\n    start_index: int = 0,\n) -> Iterator[PreparedDocument]:\n    """Yield one prepared matching document at a time.\n\n    The caller controls how far this generator is advanced. The bounded launcher\n    advances it only when a document-worker slot is available, so the parent\n    process never constructs a list of the full dev split.\n    """\n    dataset_jsonl = Path(dataset_jsonl).resolve()\n    task_guidance_path = Path(task_guidance_path).resolve()\n    batch_root = Path(batch_root).resolve()\n    selected_root = batch_root / "selected_documents"\n    selected_root.mkdir(parents=True, exist_ok=True)\n    task_guidance = read_json(task_guidance_path)\n\n    selection_jsonl = batch_root / "selection_documents.jsonl"\n    selection_jsonl.parent.mkdir(parents=True, exist_ok=True)\n    selection_jsonl.write_text("", encoding="utf-8")\n\n    selected_count = 0\n    matching_index = 0\n    for source_index, record in enumerate(iter_jsonl(dataset_jsonl)):\n        if not is_exact_type(record, required_type):\n            continue\n        if matching_index < int(start_index):\n            matching_index += 1\n            continue\n        if not run_all_documents and selected_count >= int(smoke_document_limit):\n            break\n\n        doc_id = document_id(record, source_index)\n        slug = safe_slug(doc_id)\n        selection_index = selected_count\n        doc_root = selected_root / f"{selection_index:06d}_{slug}"\n        doc_root.mkdir(parents=True, exist_ok=True)\n\n        # Gold is stripped before the native pipeline input is serialized.\n        input_record = strip_gold(record, task_guidance)\n        input_line = json.dumps(input_record, ensure_ascii=False, separators=(",", ":"))\n        gold_line = json.dumps(record, ensure_ascii=False, separators=(",", ":"))\n        input_path = doc_root / "input.jsonl"\n        gold_path = doc_root / "gold.jsonl"\n        input_path.write_text(input_line + "\\n", encoding="utf-8")\n        gold_path.write_text(gold_line + "\\n", encoding="utf-8")\n\n        prepared = PreparedDocument(\n            selection_index=selection_index,\n            source_index=source_index,\n            document_id=doc_id,\n            title=str(record.get("title")) if record.get("title") is not None else None,\n            slug=slug,\n            input_jsonl=str(input_path),\n            gold_jsonl=str(gold_path),\n            run_dir=str(batch_root / "document_runs" / f"{selection_index:06d}_{slug}"),\n            input_sha256=sha256_bytes((input_line + "\\n").encode("utf-8")),\n            gold_sha256=sha256_bytes((gold_line + "\\n").encode("utf-8")),\n        )\n        append_jsonl(selection_jsonl, asdict(prepared))\n        yield prepared\n\n        selected_count += 1\n        matching_index += 1\n\n\ndef _load_parent_resumable_result(\n    job: dict[str, Any],\n    config: BatchRunConfig,\n) -> dict[str, Any] | None:\n    if not config.resume_completed:\n        return None\n    run_dir = Path(job["run_dir"]).resolve()\n    result_path = run_dir / "document_result.json"\n    if not result_path.is_file():\n        return None\n    try:\n        previous = read_json(result_path)\n        fingerprint = base._scientific_fingerprint(job, config)\n    except Exception:\n        return None\n    if previous.get("status") != "completed":\n        return None\n    if previous.get("scientific_fingerprint") != fingerprint:\n        return None\n    resumed = dict(previous)\n    resumed["status"] = "skipped_completed"\n    resumed["resumed_at"] = time.strftime("%Y-%m-%d %H:%M:%S")\n    return resumed\n\n\ndef _persist_parent_failure(job: dict[str, Any], config: BatchRunConfig, exc: BaseException) -> dict[str, Any]:\n    run_dir = Path(job["run_dir"]).resolve()\n    run_dir.mkdir(parents=True, exist_ok=True)\n    failure = {\n        "status": "failed_parent_process",\n        "batch_version": base.BATCH_VERSION,\n        "orchestrator_version": ORCHESTRATOR_VERSION,\n        "scientific_fingerprint": base._scientific_fingerprint(job, config),\n        "selection_index": job["selection_index"],\n        "source_index": job["source_index"],\n        "document_id": job["document_id"],\n        "title": job.get("title"),\n        "slug": job["slug"],\n        "run_dir": str(run_dir),\n        "transient": False,\n        "error_type": type(exc).__name__,\n        "error": str(exc),\n        "traceback": traceback.format_exc(),\n        "failed_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(run_dir / "document_failure.json", failure)\n    return failure\n\n\ndef _event_from_result(\n    result: dict[str, Any],\n    *,\n    completed: int,\n    total: int,\n    started: float,\n) -> dict[str, Any]:\n    return {\n        "completed": completed,\n        "total": total,\n        "status": result.get("status"),\n        "document_id": result.get("document_id"),\n        "pipeline_seconds": result.get("pipeline_seconds"),\n        "pipeline_reused": result.get("pipeline_reused", False),\n        "attempt": result.get("attempt"),\n        "transient": result.get("transient"),\n        "error_type": result.get("error_type"),\n        "error": result.get("error"),\n        "elapsed_batch_seconds": round(time.time() - started, 3),\n        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n\n\ndef _print_progress(event: dict[str, Any], *, progress_every: int) -> None:\n    completed = int(event["completed"])\n    total = int(event["total"])\n    status = str(event.get("status") or "")\n    should_print = (\n        completed == 1\n        or completed == total\n        or completed % max(1, int(progress_every)) == 0\n        or status.startswith("failed")\n    )\n    if not should_print:\n        return\n    pipeline = event.get("pipeline_seconds")\n    pipeline_text = f"{pipeline}s" if pipeline is not None else "n/a"\n    suffix = " | reused saved pipeline" if event.get("pipeline_reused") else ""\n    if status.startswith("failed"):\n        suffix += (\n            f" | {event.get(\'error_type\')}: {event.get(\'error\')}"\n            f" | transient={event.get(\'transient\')}"\n        )\n    print(\n        f"[{completed}/{total}] {status}: {event.get(\'document_id\')} "\n        f"| pipeline={pipeline_text}{suffix}"\n    )\n\n\ndef run_documents_bounded_streaming(\n    *,\n    documents: Iterator[PreparedDocument],\n    expected_total: int,\n    config: BatchRunConfig,\n    api_key: str,\n    batch_root: str | Path,\n) -> dict[str, Any]:\n    """Run with at most ``document_workers`` submitted jobs in parent memory."""\n    if not api_key:\n        raise ValueError("OPENROUTER_API_KEY is not set")\n    if not 1 <= int(config.document_workers) <= 5:\n        raise ValueError("document_workers must be between 1 and 5")\n    if int(config.layer_workers) < 1:\n        raise ValueError("layer_workers must be positive")\n\n    batch_root = Path(batch_root).resolve()\n    batch_root.mkdir(parents=True, exist_ok=True)\n    events_path = batch_root / "batch_events.jsonl"\n    events_path.write_text("", encoding="utf-8")\n\n    started = time.time()\n    prepared = 0\n    completed = 0\n    completed_ok = 0\n    failed = 0\n    resumed = 0\n    exhausted = False\n    config_dict = asdict(config)\n    pending: dict[Any, dict[str, Any]] = {}\n\n    def record_result(result: dict[str, Any]) -> None:\n        nonlocal completed, completed_ok, failed, resumed\n        completed += 1\n        status = str(result.get("status") or "")\n        if status in {"completed", "skipped_completed"}:\n            completed_ok += 1\n        else:\n            failed += 1\n        if status == "skipped_completed" or result.get("pipeline_reused"):\n            resumed += 1\n        event = _event_from_result(\n            result,\n            completed=completed,\n            total=expected_total,\n            started=started,\n        )\n        append_jsonl(events_path, event)\n        _print_progress(event, progress_every=config.progress_every)\n\n    context = mp.get_context("spawn")\n    with ProcessPoolExecutor(max_workers=config.document_workers, mp_context=context) as pool:\n        while not exhausted or pending:\n            # Advance the JSONL generator only while a worker slot is available.\n            while not exhausted and len(pending) < int(config.document_workers):\n                try:\n                    prepared_document = next(documents)\n                except StopIteration:\n                    exhausted = True\n                    break\n                prepared += 1\n                job = asdict(prepared_document)\n\n                parent_resume = _load_parent_resumable_result(job, config)\n                if parent_resume is not None:\n                    record_result(parent_resume)\n                    continue\n\n                future = pool.submit(base._worker_run_document, job, config_dict, api_key)\n                pending[future] = job\n\n            if not pending:\n                continue\n\n            done, _ = wait(tuple(pending), return_when=FIRST_COMPLETED)\n            for future in done:\n                job = pending.pop(future)\n                try:\n                    result = future.result()\n                except BaseException as exc:\n                    result = _persist_parent_failure(job, config, exc)\n                record_result(result)\n\n    return {\n        "expected_total": expected_total,\n        "documents_prepared": prepared,\n        "documents_finished": completed,\n        "documents_completed_or_resumed": completed_ok,\n        "documents_failed": failed,\n        "documents_resumed": resumed,\n        "max_pending_documents": int(config.document_workers),\n        "elapsed_seconds": round(time.time() - started, 3),\n    }\n\n\ndef _metric_accumulator() -> dict[str, int]:\n    return {"predicted": 0, "gold": 0, "true_positive": 0, "false_positive": 0, "false_negative": 0}\n\n\ndef _add_metrics(acc: dict[str, int], row: dict[str, Any]) -> None:\n    for key in acc:\n        acc[key] += int(row.get(key, 0) or 0)\n\n\ndef _finish_metrics(acc: dict[str, int]) -> dict[str, Any]:\n    predicted = acc["predicted"]\n    gold = acc["gold"]\n    tp = acc["true_positive"]\n    precision = tp / predicted if predicted else 0.0\n    recall = tp / gold if gold else 0.0\n    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n    return {**acc, "precision": precision, "recall": recall, "f1": f1}\n\n\ndef _write_csv_rows(path: Path, rows: list[dict[str, Any]]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    if not rows:\n        path.write_text("", encoding="utf-8")\n        return\n    fieldnames: list[str] = []\n    seen: set[str] = set()\n    for row in rows:\n        for key in row:\n            if key not in seen:\n                seen.add(key)\n                fieldnames.append(key)\n    with path.open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(handle, fieldnames=fieldnames, extrasaction="ignore")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef aggregate_batch_results_streaming(\n    *,\n    batch_root: str | Path,\n    relation_catalog_path: str | Path | None = None,\n) -> dict[str, Any]:\n    """Aggregate per-document files without loading document results into a list."""\n    batch_root = Path(batch_root).resolve()\n    selection_path = batch_root / "selection_documents.jsonl"\n    if not selection_path.is_file():\n        raise FileNotFoundError(f"Missing selection stream: {selection_path}")\n\n    analysis_root = batch_root / "aggregate_analysis"\n    analysis_root.mkdir(parents=True, exist_ok=True)\n    per_document_csv = analysis_root / "per_document_metrics.csv"\n    per_document_jsonl = analysis_root / "document_results_compact.jsonl"\n    predictions_jsonl = analysis_root / "predictions.jsonl"\n    failures_jsonl = analysis_root / "failed_documents.jsonl"\n\n    per_document_fields = [\n        "selection_index", "source_index", "document_id", "title", "status",\n        "pipeline_seconds", "wall_seconds", "relation_predicted", "relation_gold",\n        "relation_tp", "relation_fp", "relation_fn", "relation_precision",\n        "relation_recall", "relation_f1", "entity_precision", "entity_recall",\n        "entity_f1", "endpoint_precision", "endpoint_recall", "endpoint_f1", "run_dir",\n    ]\n\n    relation_acc = _metric_accumulator()\n    entity_acc = _metric_accumulator()\n    endpoint_acc = _metric_accumulator()\n    macro_relation_sums = {"precision": 0.0, "recall": 0.0, "f1": 0.0}\n    macro_entity_sums = {"precision": 0.0, "recall": 0.0, "f1": 0.0}\n    relation_counts: dict[str, dict[str, int]] = {}\n    failure_counts: dict[str, int] = {}\n    layer_buckets: dict[int, dict[str, Any]] = {}\n    pipeline_seconds_values: list[float] = []  # numeric only; no document payloads\n    failed_preview: list[dict[str, Any]] = []\n\n    requested = 0\n    completed_count = 0\n    failed_count = 0\n\n    with (\n        per_document_csv.open("w", encoding="utf-8", newline="") as csv_handle,\n        per_document_jsonl.open("w", encoding="utf-8") as compact_handle,\n        predictions_jsonl.open("w", encoding="utf-8") as predictions_handle,\n        failures_jsonl.open("w", encoding="utf-8") as failures_handle,\n    ):\n        csv_writer = csv.DictWriter(csv_handle, fieldnames=per_document_fields, extrasaction="ignore")\n        csv_writer.writeheader()\n\n        for job in iter_jsonl(selection_path):\n            requested += 1\n            run_dir = Path(job["run_dir"]).resolve()\n            result_path = run_dir / "document_result.json"\n            failure_path = run_dir / "document_failure.json"\n\n            result: dict[str, Any] | None = None\n            if result_path.is_file():\n                candidate = read_json(result_path)\n                if candidate.get("status") == "completed":\n                    result = candidate\n\n            if result is None:\n                failed_count += 1\n                if failure_path.is_file():\n                    failure = read_json(failure_path)\n                else:\n                    failure = {\n                        "status": "missing_result",\n                        "selection_index": job.get("selection_index"),\n                        "source_index": job.get("source_index"),\n                        "document_id": job.get("document_id"),\n                        "title": job.get("title"),\n                        "run_dir": str(run_dir),\n                        "error_type": "MissingResult",\n                        "error": "No completed document_result.json or document_failure.json was found.",\n                    }\n                failures_handle.write(json.dumps(failure, ensure_ascii=False, default=_json_default) + "\\n")\n                if len(failed_preview) < 50:\n                    failed_preview.append(failure)\n                continue\n\n            completed_count += 1\n            rel = result["relation_metrics"]\n            ent = result["entity_metrics"]["entity_inventory"]\n            endpoint = result["entity_metrics"]["relation_endpoint_inventory"]\n            _add_metrics(relation_acc, rel)\n            _add_metrics(entity_acc, ent)\n            _add_metrics(endpoint_acc, endpoint)\n            for key in macro_relation_sums:\n                macro_relation_sums[key] += float(rel.get(key, 0.0) or 0.0)\n                macro_entity_sums[key] += float(ent.get(key, 0.0) or 0.0)\n\n            pipeline_seconds = float(result.get("pipeline_seconds") or 0.0)\n            pipeline_seconds_values.append(pipeline_seconds)\n            row = {\n                "selection_index": result.get("selection_index"),\n                "source_index": result.get("source_index"),\n                "document_id": result.get("document_id"),\n                "title": result.get("title"),\n                "status": result.get("status"),\n                "pipeline_seconds": result.get("pipeline_seconds"),\n                "wall_seconds": result.get("wall_seconds"),\n                "relation_predicted": rel.get("predicted"),\n                "relation_gold": rel.get("gold"),\n                "relation_tp": rel.get("true_positive"),\n                "relation_fp": rel.get("false_positive"),\n                "relation_fn": rel.get("false_negative"),\n                "relation_precision": rel.get("precision"),\n                "relation_recall": rel.get("recall"),\n                "relation_f1": rel.get("f1"),\n                "entity_precision": ent.get("precision"),\n                "entity_recall": ent.get("recall"),\n                "entity_f1": ent.get("f1"),\n                "endpoint_precision": endpoint.get("precision"),\n                "endpoint_recall": endpoint.get("recall"),\n                "endpoint_f1": endpoint.get("f1"),\n                "run_dir": result.get("run_dir"),\n            }\n            csv_writer.writerow(row)\n            compact_handle.write(json.dumps(row, ensure_ascii=False, default=_json_default) + "\\n")\n\n            for prediction in result.get("predictions", []):\n                predictions_handle.write(json.dumps({\n                    "document_id": result.get("document_id"),\n                    "title": result.get("title"),\n                    **prediction,\n                }, ensure_ascii=False, default=_json_default) + "\\n")\n\n            for bucket in ("tp", "fp", "fn"):\n                for triple in rel.get(bucket, []):\n                    if len(triple) != 3:\n                        continue\n                    relation_id = str(triple[1])\n                    counts = relation_counts.setdefault(relation_id, {"tp": 0, "fp": 0, "fn": 0})\n                    counts[bucket] += 1\n\n            for reason, count in (result.get("failure_counts") or {}).items():\n                failure_counts[str(reason)] = failure_counts.get(str(reason), 0) + int(count)\n\n            for layer_row in result.get("cumulative_evaluation", []):\n                index = int(layer_row.get("layer_index", -1))\n                bucket = layer_buckets.setdefault(index, {\n                    "layer_index": index,\n                    "layer_name": layer_row.get("layer_name"),\n                    "predicted": 0,\n                    "gold": 0,\n                    "true_positive": 0,\n                    "false_positive": 0,\n                    "false_negative": 0,\n                })\n                for key in ["predicted", "gold", "true_positive", "false_positive", "false_negative"]:\n                    bucket[key] += int(layer_row.get(key, 0) or 0)\n\n    labels: dict[str, str] = {}\n    if relation_catalog_path and Path(relation_catalog_path).is_file():\n        catalog = read_json(relation_catalog_path)\n        for relation in catalog.get("properties", []):\n            relation_id = str(relation.get("property_id") or relation.get("id") or "")\n            if relation_id:\n                labels[relation_id] = str(relation.get("label") or "")\n\n    per_relation: list[dict[str, Any]] = []\n    for relation_id, counts in sorted(relation_counts.items()):\n        predicted = counts["tp"] + counts["fp"]\n        gold = counts["tp"] + counts["fn"]\n        precision = counts["tp"] / predicted if predicted else 0.0\n        recall = counts["tp"] / gold if gold else 0.0\n        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n        per_relation.append({\n            "relation_id": relation_id,\n            "label": labels.get(relation_id, ""),\n            "predicted": predicted,\n            "gold": gold,\n            "true_positive": counts["tp"],\n            "false_positive": counts["fp"],\n            "false_negative": counts["fn"],\n            "precision": precision,\n            "recall": recall,\n            "f1": f1,\n        })\n\n    cumulative: list[dict[str, Any]] = []\n    for index in sorted(layer_buckets):\n        row = layer_buckets[index]\n        precision = row["true_positive"] / row["predicted"] if row["predicted"] else 0.0\n        recall = row["true_positive"] / row["gold"] if row["gold"] else 0.0\n        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0\n        cumulative.append({**row, "precision": precision, "recall": recall, "f1": f1})\n\n    micro_relation = _finish_metrics(relation_acc)\n    micro_entity = _finish_metrics(entity_acc)\n    micro_endpoint = _finish_metrics(endpoint_acc)\n    macro_relation = {\n        key: macro_relation_sums[key] / completed_count if completed_count else 0.0\n        for key in macro_relation_sums\n    }\n    macro_entity = {\n        key: macro_entity_sums[key] / completed_count if completed_count else 0.0\n        for key in macro_entity_sums\n    }\n    total_pipeline_seconds = sum(pipeline_seconds_values)\n\n    summary = {\n        "batch_version": base.BATCH_VERSION,\n        "orchestrator_version": ORCHESTRATOR_VERSION,\n        "required_type_key": "type",\n        "required_type_value": EXACT_REQUIRED_TYPE,\n        "documents_requested": requested,\n        "documents_completed_or_resumed": completed_count,\n        "documents_failed": failed_count,\n        "micro_relation": micro_relation,\n        "macro_relation": macro_relation,\n        "micro_entity_inventory": micro_entity,\n        "macro_entity_inventory": macro_entity,\n        "micro_relation_endpoint_inventory": micro_endpoint,\n        "total_document_pipeline_seconds": total_pipeline_seconds,\n        "mean_document_pipeline_seconds": total_pipeline_seconds / completed_count if completed_count else 0.0,\n        "median_document_pipeline_seconds": statistics.median(pipeline_seconds_values) if pipeline_seconds_values else 0.0,\n        "first_failure_counts": failure_counts,\n        "per_document_metrics_csv": str(per_document_csv),\n        "failed_documents_jsonl": str(failures_jsonl),\n        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n\n    write_json(analysis_root / "batch_summary.json", summary)\n    write_json(analysis_root / "per_relation_metrics.json", per_relation)\n    write_json(analysis_root / "cumulative_layer_micro_evaluation.json", cumulative)\n    _write_csv_rows(analysis_root / "per_relation_metrics.csv", per_relation)\n    _write_csv_rows(analysis_root / "cumulative_layer_micro_evaluation.csv", cumulative)\n\n    return {\n        "summary": summary,\n        "per_relation": per_relation,\n        "cumulative": cumulative,\n        "failed_preview": failed_preview,\n        "paths": {\n            "per_document_csv": str(per_document_csv),\n            "per_document_jsonl": str(per_document_jsonl),\n            "predictions_jsonl": str(predictions_jsonl),\n            "failed_documents_jsonl": str(failures_jsonl),\n            "batch_summary": str(analysis_root / "batch_summary.json"),\n        },\n    }\n\n\ndef run_batch_streaming(\n    *,\n    dataset_jsonl: str | Path,\n    task_guidance_path: str | Path,\n    batch_root: str | Path,\n    run_all_documents: bool,\n    smoke_document_limit: int,\n    start_index: int,\n    config: BatchRunConfig,\n    api_key: str,\n    matching_record_count: int | None = None,\n    required_type: str = EXACT_REQUIRED_TYPE,\n) -> dict[str, Any]:\n    batch_root = Path(batch_root).resolve()\n    batch_root.mkdir(parents=True, exist_ok=True)\n\n    if matching_record_count is None:\n        matching_record_count = int(count_exact_type_records(\n            dataset_jsonl,\n            required_type=required_type,\n            first_n_ids=0,\n        )["matching_records"])\n    available = max(0, int(matching_record_count) - int(start_index))\n    selected_count = available if run_all_documents else min(int(smoke_document_limit), available)\n    if selected_count <= 0:\n        raise ValueError(\n            f"No records with exact key/value type={required_type!r} remain after start_index={start_index}."\n        )\n\n    selection_manifest = {\n        "orchestrator_version": ORCHESTRATOR_VERSION,\n        "scientific_batch_version": base.BATCH_VERSION,\n        "dataset_jsonl": str(Path(dataset_jsonl).resolve()),\n        "dataset_sha256": sha256_file(dataset_jsonl),\n        "task_guidance_path": str(Path(task_guidance_path).resolve()),\n        "task_guidance_sha256": sha256_file(task_guidance_path),\n        "filter": {"key": "type", "value": required_type, "fallback_to_split": False},\n        "run_all_documents": run_all_documents,\n        "smoke_document_limit": smoke_document_limit,\n        "start_index": start_index,\n        "matching_record_count": matching_record_count,\n        "selected_count": selected_count,\n        "selection_documents_jsonl": str(batch_root / "selection_documents.jsonl"),\n        "memory_policy": {\n            "full_document_list_materialized": False,\n            "maximum_pending_document_jobs": int(config.document_workers),\n        },\n        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(batch_root / "selection_manifest.json", selection_manifest)\n\n    invocation = {\n        **selection_manifest,\n        "config": {**asdict(config), "api_key": "NOT_STORED"},\n        "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n    }\n    write_json(batch_root / "batch_invocation.json", invocation)\n\n    documents = iter_prepared_documents_streaming(\n        dataset_jsonl=dataset_jsonl,\n        task_guidance_path=task_guidance_path,\n        batch_root=batch_root,\n        run_all_documents=run_all_documents,\n        smoke_document_limit=smoke_document_limit,\n        required_type=required_type,\n        start_index=start_index,\n    )\n    scheduler = run_documents_bounded_streaming(\n        documents=documents,\n        expected_total=selected_count,\n        config=config,\n        api_key=api_key,\n        batch_root=batch_root,\n    )\n    aggregate = aggregate_batch_results_streaming(\n        batch_root=batch_root,\n        relation_catalog_path=config.relation_catalog_path,\n    )\n\n    invocation["completed_at"] = time.strftime("%Y-%m-%d %H:%M:%S")\n    invocation["scheduler"] = scheduler\n    invocation["documents_completed_or_resumed"] = aggregate["summary"]["documents_completed_or_resumed"]\n    invocation["documents_failed"] = aggregate["summary"]["documents_failed"]\n    write_json(batch_root / "batch_invocation.json", invocation)\n    return {"scheduler": scheduler, **aggregate}\n'
    V62_HELPER.write_text(_embedded_v62, encoding="utf-8")
    print("Installed missing helper:", V62_HELPER)
else:
    print("Using existing helper:", V62_HELPER)

# v6.2 intentionally builds on the frozen v6.1 scientific runner.
# Search all likely locations before importing.
v61_candidates = [
    TOOLS_DIR / "docred_native_batch_v6_1.py",
    NOTEBOOK_DIR / "docred_native_batch_v6_1.py",
    PROJECT_ROOT / "docred_native_batch_v6_1.py",
]
v61_found = next((p for p in v61_candidates if p.is_file()), None)
if v61_found is None:
    raise FileNotFoundError(
        "The v6.2 helper is now present, but its frozen dependency "
        "docred_native_batch_v6_1.py is missing. Checked:\n"
        + "\n".join(str(p) for p in v61_candidates)
        + "\n\nIf you see this error, send it to me; the missing file is "
          "docred_native_batch_v6_1.py, not the v6.2 helper."
    )

if str(v61_found.parent) not in sys.path:
    sys.path.insert(0, str(v61_found.parent))

# Clear a stale failed import if this cell is rerun in the same kernel.
sys.modules.pop("docred_native_batch_v6_2_dev_streaming", None)

from docred_native_batch_v6_2_dev_streaming import (
    BatchRunConfig,
    aggregate_batch_results_streaming,
    count_exact_type_records,
    read_json,
    run_batch_streaming,
)

mp.freeze_support()

print("PROJECT_ROOT =", PROJECT_ROOT)
print("TOOLS_DIR    =", TOOLS_DIR)
print("v6.1 helper  =", v61_found)
print("v6.2 helper  =", V62_HELPER)


Installed missing helper: C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\tools\docred_native_batch_v6_2_dev_streaming.py


FileNotFoundError: The v6.2 helper is now present, but its frozen dependency docred_native_batch_v6_1.py is missing. Checked:
C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\tools\docred_native_batch_v6_1.py
C:\Users\galencarmedeiro\NeoOLAF\examples\RAGTreeDatasets\docred_native_batch_v6_1.py
C:\Users\galencarmedeiro\NeoOLAF\docred_native_batch_v6_1.py

If you see this error, send it to me; the missing file is docred_native_batch_v6_1.py, not the v6.2 helper.

## Frozen DocRED paths and execution configuration

These values match the original v6.2 full-dev runner. The same batch root is critical: this is what lets the scheduler recognize and skip the already completed 978 documents.


In [ ]:
RUN_ALL_DEV_DOCUMENTS = True
EXACT_TYPE_VALUE = "dev"
START_DEV_INDEX = 0

# Only used by the helper API when run_all_documents=False.
SMOKE_DEV_DOCUMENT_LIMIT = 5

DATASET_JSONL = first_existing_path(
    "DATASET_JSONL",
    [
        NOTEBOOK_DIR / "../../../ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "ragtree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT.parent / "RAGTree/data/preprocessed/docred_causal.jsonl",
        PROJECT_ROOT / "ragtree/data/preprocessed/docred_causal.jsonl",
    ],
)

# MUST stay identical to the original v6.2 full run.
BATCH_ROOT = NOTEBOOK_DIR / "runs/docred_native_v5_1_dev_streaming"

ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"
PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation_v5.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_v5.json"
TASK_GUIDANCE_PATH = NOTEBOOK_DIR / "configs/docred_task_guidance_v5_1_frozen.json"

OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"

# Original v6.2 orchestration.
DOCUMENT_WORKERS = 4
LAYER_WORKERS = 16

REASONING_EFFORT = "minimal"
MAX_TOKENS = 4096
REQUEST_TIMEOUT = 120

# These are the critical resume settings.
RESUME_COMPLETED = True
RETRY_FAILED_DOCUMENTS = True

# Same retry behavior as the original runner.
DOCUMENT_ATTEMPTS = 2
RETRY_BACKOFF_SECONDS = 8.0
DOCUMENT_LAUNCH_STAGGER_SECONDS = 0.75
VERBOSE_DOCUMENTS = False
PROGRESS_EVERY = 1

# Paid execution switch.
RUN_RETRY = True

print("BATCH_ROOT =", BATCH_ROOT)
print("MODEL =", MODEL_NAME)
print("DOCUMENT_WORKERS =", DOCUMENT_WORKERS)
print("LAYER_WORKERS =", LAYER_WORKERS)
print("RESUME_COMPLETED =", RESUME_COMPLETED)
print("RETRY_FAILED_DOCUMENTS =", RETRY_FAILED_DOCUMENTS)


## Zero-cost scientific and filesystem preflight


In [ ]:
required = [
    DATASET_JSONL,
    ONTOLOGY_PATH,
    ONTOLOGY_ORIGINAL,
    RELATION_CATALOG,
    RELATION_ALIASES,
    PROFILE_PATH,
    GUIDANCE_PATH,
    TASK_GUIDANCE_PATH,
]

missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

if not BATCH_ROOT.is_dir():
    raise FileNotFoundError(
        "The original DocRED batch root does not exist. "
        "This notebook is resume-only and refuses to create a new experiment:\n"
        f"{BATCH_ROOT}"
    )

profile = read_json(PROFILE_PATH)
task_guidance = read_json(TASK_GUIDANCE_PATH)
catalog = read_json(RELATION_CATALOG)

corpus_scan = count_exact_type_records(
    DATASET_JSONL,
    required_type=EXACT_TYPE_VALUE,
    first_n_ids=5,
)

matching_records = int(corpus_scan["matching_records"])

print("JSONL records:", corpus_scan["total_records"])
print('Records with exact type="dev":', matching_records)
print("Ontology properties:", catalog["property_count"])
print("Allowed relation IDs:", len(task_guidance["allowed_relation_ids"]))
print("Frozen profile:", profile["profile_name"])

assert EXACT_TYPE_VALUE == "dev"
assert matching_records == 998, (
    "Expected the same 998 exact type=dev records from the original run.",
    matching_records,
)
assert catalog["property_count"] == 96
assert len(task_guidance["allowed_relation_ids"]) == 96

# Anti-cheating / gold-isolation invariants from the frozen profile.
assert profile["relations"]["allowed"] == []
assert profile["anti_cheating"]["direct_docred_extraction"] is False
assert profile["anti_cheating"]["source_entity_anchoring"] is False
assert profile["anti_cheating"]["post_run_relation_invention"] is False
assert profile["benchmark_projection"]["gold_available_to_pipeline"] is False

# Freeze the execution shell.
assert DOCUMENT_WORKERS == 4
assert LAYER_WORKERS == 16
assert RESUME_COMPLETED is True
assert RETRY_FAILED_DOCUMENTS is True

print("\nPreflight: OK")
print("No API calls made.")


## Rebuild CURRENT aggregate from saved artifacts — zero API calls

This determines the exact current number of successful and failed documents before any retry.

At the original stopping point this should show **978 completed / 20 failed**.  
The notebook also remains safe if you rerun it after recovering some of those failures.


In [ ]:
before = aggregate_batch_results_streaming(
    batch_root=BATCH_ROOT,
    relation_catalog_path=RELATION_CATALOG,
)

before_summary = before["summary"]

before_requested = int(before_summary["documents_requested"])
before_completed = int(before_summary["documents_completed_or_resumed"])
before_failed = int(before_summary["documents_failed"])

print("CURRENT DOCRED STATE")
print("====================")
print("requested:", before_requested)
print("completed/resumed:", before_completed)
print("failed:", before_failed)

assert before_requested == 998, before_requested
assert before_completed + before_failed == before_requested, (
    before_completed,
    before_failed,
    before_requested,
)
assert before_completed >= 978, (
    "This resume notebook expects the previous full run or a later partial retry.",
    before_completed,
)
assert 0 <= before_failed <= 20, before_failed

failed_path = Path(before["paths"]["failed_documents_jsonl"])
print("Failures JSONL:", failed_path)

def read_jsonl(path: Path):
    rows = []
    if not path.is_file():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

failed_rows_before = read_jsonl(failed_path)

# The aggregate count is authoritative. The failure artifact should agree.
assert len(failed_rows_before) == before_failed, (
    len(failed_rows_before),
    before_failed,
)

if failed_rows_before:
    failed_df = pd.DataFrame(failed_rows_before)
    columns = [
        c for c in [
            "selection_index",
            "source_index",
            "document_id",
            "title",
            "status",
            "error_type",
            "error",
            "run_dir",
        ]
        if c in failed_df.columns
    ]
    display(failed_df[columns])
else:
    print("No failed documents remain.")

display(Markdown(
    f"**Current state:** {before_completed}/998 complete; "
    f"**{before_failed} failed documents eligible for retry**.  \n"
    "The completed documents are protected by `resume_completed=True`."
))


## Paid retry cell — ONLY failed/unresolved documents are eligible

`run_batch_streaming` still scans the dev JSONL to recover source records, but the existing batch state causes completed records to be resumed/skipped.

It does **not** pay to rerun the already-completed documents.


In [ ]:
if before_failed == 0:
    print("DocRED is already 998/998. Paid retry skipped.")
    batch = before

elif not RUN_RETRY:
    print(
        f"RUN_RETRY=False: {before_failed} failed document(s) remain. "
        "No API calls made."
    )
    batch = before

else:
    API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip().strip('"').strip("'")
    if not API_KEY:
        API_KEY = getpass("OpenRouter API key: ").strip().strip('"').strip("'")
    if not API_KEY:
        raise RuntimeError("No OpenRouter API key was provided.")

    config = BatchRunConfig(
        project_root=str(PROJECT_ROOT),
        ontology_path=str(ONTOLOGY_PATH),
        profile_path=str(PROFILE_PATH),
        guidance_path=str(GUIDANCE_PATH),
        relation_catalog_path=str(RELATION_CATALOG),
        relation_aliases_path=str(RELATION_ALIASES),
        model_name=MODEL_NAME,
        host=OPENROUTER_HOST,
        document_workers=DOCUMENT_WORKERS,
        layer_workers=LAYER_WORKERS,
        reasoning_effort=REASONING_EFFORT,
        max_tokens=MAX_TOKENS,
        request_timeout=REQUEST_TIMEOUT,
        resume_completed=RESUME_COMPLETED,
        retry_failed_documents=RETRY_FAILED_DOCUMENTS,
        document_attempts=DOCUMENT_ATTEMPTS,
        retry_backoff_seconds=RETRY_BACKOFF_SECONDS,
        launch_stagger_seconds=DOCUMENT_LAUNCH_STAGGER_SECONDS,
        verbose_documents=VERBOSE_DOCUMENTS,
        progress_every=PROGRESS_EVERY,
    )

    print(
        f"Retrying unresolved DocRED documents only: {before_failed} currently failed.\n"
        f"Completed documents protected: {before_completed}."
    )

    batch = run_batch_streaming(
        dataset_jsonl=DATASET_JSONL,
        task_guidance_path=TASK_GUIDANCE_PATH,
        batch_root=BATCH_ROOT,
        run_all_documents=True,
        smoke_document_limit=SMOKE_DEV_DOCUMENT_LIMIT,
        start_index=0,
        config=config,
        api_key=API_KEY,
        matching_record_count=matching_records,
        required_type=EXACT_TYPE_VALUE,
    )

    print("Retry invocation finished.")


## Rebuild final aggregate from disk — zero additional API calls

This deliberately rebuilds the aggregate again from persisted document artifacts instead of trusting only the in-memory scheduler result.


In [ ]:
final_batch = aggregate_batch_results_streaming(
    batch_root=BATCH_ROOT,
    relation_catalog_path=RELATION_CATALOG,
)

summary = final_batch["summary"]

requested = int(summary["documents_requested"])
completed = int(summary["documents_completed_or_resumed"])
failed = int(summary["documents_failed"])

print("FINAL/CURRENT DOCRED STATE")
print("==========================")
print("requested:", requested)
print("completed/resumed:", completed)
print("failed:", failed)

assert requested == 998
assert completed + failed == requested

final_metrics = {
    "documents_requested": requested,
    "documents_completed_or_resumed": completed,
    "documents_failed": failed,

    "relation_micro_precision": summary["micro_relation"]["precision"],
    "relation_micro_recall": summary["micro_relation"]["recall"],
    "relation_micro_f1": summary["micro_relation"]["f1"],

    "relation_macro_precision": summary["macro_relation"]["precision"],
    "relation_macro_recall": summary["macro_relation"]["recall"],
    "relation_macro_f1": summary["macro_relation"]["f1"],

    "entity_micro_precision": summary["micro_entity_inventory"]["precision"],
    "entity_micro_recall": summary["micro_entity_inventory"]["recall"],
    "entity_micro_f1": summary["micro_entity_inventory"]["f1"],

    "endpoint_micro_precision": summary["micro_relation_endpoint_inventory"]["precision"],
    "endpoint_micro_recall": summary["micro_relation_endpoint_inventory"]["recall"],
    "endpoint_micro_f1": summary["micro_relation_endpoint_inventory"]["f1"],

    "mean_pipeline_seconds": summary["mean_document_pipeline_seconds"],
    "median_pipeline_seconds": summary["median_document_pipeline_seconds"],
}

display(pd.DataFrame([final_metrics]))

print("\nRELATION MICRO")
print(
    f"P={summary['micro_relation']['precision']:.9f} | "
    f"R={summary['micro_relation']['recall']:.9f} | "
    f"F1={summary['micro_relation']['f1']:.9f}"
)

print("\nENTITY MICRO")
print(
    f"P={summary['micro_entity_inventory']['precision']:.9f} | "
    f"R={summary['micro_entity_inventory']['recall']:.9f} | "
    f"F1={summary['micro_entity_inventory']['f1']:.9f}"
)

print("\nRELATION ENDPOINT MICRO")
print(
    f"P={summary['micro_relation_endpoint_inventory']['precision']:.9f} | "
    f"R={summary['micro_relation_endpoint_inventory']['recall']:.9f} | "
    f"F1={summary['micro_relation_endpoint_inventory']['f1']:.9f}"
)

failed_rows_after = read_jsonl(Path(final_batch["paths"]["failed_documents_jsonl"]))

if failed_rows_after:
    print(f"\nStill failed: {len(failed_rows_after)}")
    failed_after_df = pd.DataFrame(failed_rows_after)
    columns = [
        c for c in [
            "selection_index",
            "source_index",
            "document_id",
            "title",
            "status",
            "error_type",
            "error",
            "run_dir",
        ]
        if c in failed_after_df.columns
    ]
    display(failed_after_df[columns])
    print(
        "\nRerun this notebook to retry ONLY those remaining failures. "
        "Newly recovered documents will be skipped next time."
    )
else:
    print("\nSUCCESS: DocRED is now 998/998 with zero failed documents.")


## Exact relation counts + final exports

This reads the aggregate files generated by the existing DocRED evaluator, including TP/FP/FN/predicted/gold relation totals where available.


In [ ]:
per_document_path = Path(final_batch["paths"]["per_document_csv"])
per_document = (
    pd.read_csv(per_document_path)
    if per_document_path.is_file()
    else pd.DataFrame()
)

cumulative_path = BATCH_ROOT / "aggregate_analysis/cumulative_layer_micro_evaluation.csv"
cumulative = (
    pd.read_csv(cumulative_path)
    if cumulative_path.is_file()
    else pd.DataFrame()
)

if not cumulative.empty:
    final_layer = cumulative.sort_values("layer_index").iloc[-1]
    print("FINAL LAYER RELATION COUNTS")
    for key in [
        "predicted",
        "gold",
        "true_positive",
        "false_positive",
        "false_negative",
        "precision",
        "recall",
        "f1",
    ]:
        if key in final_layer:
            print(f"{key}: {final_layer[key]}")

if not per_document.empty:
    print("\nPER-DOCUMENT RUNTIME")
    if "pipeline_seconds" in per_document.columns:
        print("documents:", len(per_document))
        print("mean:", float(per_document["pipeline_seconds"].mean()))
        print("median:", float(per_document["pipeline_seconds"].median()))
        print("min:", float(per_document["pipeline_seconds"].min()))
        print("max:", float(per_document["pipeline_seconds"].max()))

FINAL_REPORT_PATH = BATCH_ROOT / "aggregate_analysis/docred_final_resume20_report.json"
FINAL_REPORT_PATH.write_text(
    json.dumps(
        {
            "model": MODEL_NAME,
            "batch_root": str(BATCH_ROOT),
            "retry_started_from": {
                "completed": before_completed,
                "failed": before_failed,
            },
            "final": final_metrics,
            "remaining_failed_documents": failed_rows_after,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print("\nSaved final report:", FINAL_REPORT_PATH)
print("Batch summary:", final_batch["paths"]["batch_summary"])
print("Per-document CSV:", final_batch["paths"]["per_document_csv"])
print("Per-relation CSV:", BATCH_ROOT / "aggregate_analysis/per_relation_metrics.csv")
print("Cumulative layer CSV:", cumulative_path)
print("Failures JSONL:", final_batch["paths"]["failed_documents_jsonl"])
